# Лабораторная работа 7. Динамика обучения трансформера (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 3:** предобучение, оптимизация, масштабирование

**Цель.** Понять, от чего зависят устойчивость и скорость обучения трансформера: выбор оптимизатора, learning rate, расписание, параметризация.

**Что нужно сдать:**
1. Собственную реализацию AdamW, совпадающую с `torch.optim.AdamW`.
2. LR-sweep для SGD+momentum, Adam и AdamW с границей устойчивости.
3. Сравнение расписаний: constant, warmup + cosine, WSD.
4. Диагностику всплесков loss: нормы градиентов, максимальные логиты attention, статистики Adam.
5. Упрощённый μP по ширине и проверку переноса оптимального learning rate.

**Теория. AdamW.** Для градиента \(g_t\):
\[
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,\quad
\hat m_t = \frac{m_t}{1-\beta_1^t},\quad \hat v_t = \frac{v_t}{1-\beta_2^t},
\]
\[
\theta_t = \theta_{t-1} - \eta\,\lambda\,\theta_{t-1} - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t} + \varepsilon}.
\]
В отличие от L2-регуляризации в Adam, затухание весов не проходит через адаптивный масштаб \(1/\sqrt{\hat v_t}\) (Loshchilov & Hutter, 2019). Bias correction компенсирует инициализацию моментов нулём; без неё первые шаги имеют неверный масштаб.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Эталонный mini-GPT (решение ЛР 6), используется как готовый модуль ====
from dataclasses import dataclass, asdict

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text)); self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids): return "".join(self.chars[i] for i in ids)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rope_cache(T, d_head, base=10000.0, device="cpu", scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(T, device=device).float() / scale
    ang = torch.outer(t, inv)                        # (T, d_head/2)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                         # x: (B, H, T, d_head)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.n2, self.mlp = RMSNorm(cfg.n_embd), SwiGLU(cfg.n_embd, int(8 * cfg.n_embd / 3))
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        cos, sin = rope_cache(4 * cfg.block_size, cfg.n_embd // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():                  # масштабирование выходных проекций по глубине
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def get_batch(data, block_size, batch_size, device=DEVICE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20):
    model.eval(); losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train(); return float(np.mean(losses))

def train_lm(model, train_data, val_data, steps, lr=3e-3, batch_size=32, warmup=50, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    hist = []
    for s in range(steps):
        x, y = get_batch(train_data, model.cfg.block_size, batch_size)
        _, loss = model(x, y); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if s % log_every == 0 or s == steps - 1:
            hist.append((s, loss.item(), estimate_loss(model, val_data, model.cfg.block_size)))
    return hist

def get_reference_gpt(text, cfg=None, steps=None, name=None):
    """Загружает или обучает эталонную символьную модель на тексте."""
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
    cfg = cfg or (GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 256, 6, 6, 384))
    name = name or f"ref_gpt_L{cfg.n_layer}_d{cfg.n_embd}_T{cfg.block_size}.pt"
    path = ARTIFACTS / name
    model = GPT(cfg).to(DEVICE)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    else:
        train_lm(model, train_data, val_data, steps or (300 if FAST_DEV_RUN else 5000))
        torch.save(model.state_dict(), path)
    return model, tok, train_data, val_data

In [ ]:
text = load_shakespeare()
tok = CharTokenizer(text); data = torch.tensor(tok.encode(text)); ntr = int(0.9 * len(data))
train_data, val_data = data[:ntr], data[ntr:]
SMALL = GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 128, 4, 4, 128)
SWEEP_STEPS = 150 if FAST_DEV_RUN else 1500

## 1. AdamW своими руками

Реализуйте класс-наследник `torch.optim.Optimizer`. Проверьте совпадение параметров с `torch.optim.AdamW` после 20 шагов на одной и той же модели и данных (допуск 1e-6). Сравнение ведётся в float64: в float32 различия порядка 1e-7 из-за порядка операций усиливаются нормировкой \(1/\sqrt{\hat v}\) для редко обновляемых параметров и через 20 шагов достигают 1e-2.

In [ ]:
class MyAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            b1, b2 = group["betas"]
            for p in group["params"]:
                if p.grad is None: continue
                st = self.state[p]
                # TODO: ваш код
                raise NotImplementedError

def run_steps(opt_cls, n=20, **kw):
    set_seed(0); m = GPT(SMALL).to(DEVICE).double(); opt = opt_cls(m.parameters(), **kw)   # float64: Adam усиливает ошибки округления
    g = torch.Generator().manual_seed(1)
    for _ in range(n):
        ix = torch.randint(len(train_data) - 65, (8,), generator=g)
        x = torch.stack([train_data[i:i + 64] for i in ix]).to(DEVICE); y = torch.stack([train_data[i + 1:i + 65] for i in ix]).to(DEVICE)
        loss = m(x, y)[1]; opt.zero_grad(); loss.backward(); opt.step()
    return m

kw = dict(lr=3e-3, betas=(0.9, 0.95), weight_decay=0.1)
a, b = run_steps(MyAdamW, **kw), run_steps(torch.optim.AdamW, **kw)
diff = max((p - q).abs().max().item() for p, q in zip(a.parameters(), b.parameters()))
print("max |θ_my − θ_torch| =", diff); assert diff < 1e-5

## 2. LR-sweep для трёх оптимизаторов

Для SGD+momentum (0.9), Adam (L2-регуляризация через `weight_decay`) и AdamW проведите sweep по сетке learning rate (логарифмической), фиксируя всё остальное (warmup 10% шагов, затем cosine). Постройте итоговый val loss в зависимости от LR. Граница устойчивости — наименьший LR, при котором loss расходится (NaN или больше начального).

In [ ]:
def train_run(cfg, opt_name, lr, steps, schedule="cosine", warmup_frac=0.1, wd=0.1, B=32, seed=SEED, log_diag=False, param_groups=None, model=None):
    set_seed(seed); model = model or GPT(cfg).to(DEVICE)
    params = param_groups(model, lr) if param_groups else [{"params": model.parameters(), "lr": lr}]
    if opt_name == "sgd": opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=0.0)
    elif opt_name == "adam": opt = torch.optim.Adam(params, lr=lr, betas=(0.9, 0.95), weight_decay=wd * 1e-2)
    else: opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=wd)
    warm = int(warmup_frac * steps)
    # TODO: ваш код (множитель LR как функция шага для "constant", "cosine" и "wsd" (последние 20% — линейный спад))
    raise NotImplementedError
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_mult)
    hist, diag = [], []
    for s in range(steps):
        x, y = get_batch(train_data, cfg.block_size, B)
        loss = model(x, y)[1]
        if not torch.isfinite(loss): hist.append((s, float("nan"))); break
        opt.zero_grad(set_to_none=True); loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9 if log_diag else 1.0)
        if log_diag: diag.append(diagnostics(model, opt, x, gn.item(), loss.item()))
        opt.step(); sched.step(); hist.append((s, loss.item()))
    val = estimate_loss(model, val_data, cfg.block_size, iters=10) if math.isfinite(hist[-1][1]) else float("nan")
    return model, hist, val, diag

LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
# TODO: ваш код (sweep для sgd/adam/adamw, график val loss(LR))

## 3. Расписания

Для AdamW с лучшим LR сравните три расписания на одинаковом числе шагов: constant (с warmup), warmup + cosine, WSD (warmup–stable–decay с линейным спадом на последних 20% шагов). Постройте кривые train loss (скользящее среднее) и объясните характерную ступеньку WSD в фазе спада.

In [ ]:
best_lr = min((v, lr) for (o, lr), v in sweep.items() if o == "adamw" and math.isfinite(v))[1]
plt.figure(figsize=(7, 4))
for sch in ("constant", "cosine", "wsd"):
    _, h, val, _ = train_run(SMALL, "adamw", best_lr, 2 * SWEEP_STEPS, schedule=sch)
    l = np.array([v for _, v in h]); sm = np.convolve(l, np.ones(15) / 15, mode="valid")
    plt.plot(sm, label=f"{sch} (val {val:.3f})")
plt.xlabel("шаг"); plt.ylabel("train loss (скользящее среднее)"); plt.legend(); plt.show()

## 4. Всплески loss и их диагностика

Запустите AdamW с LR в 3–10 раз больше оптимального **без warmup** и без gradient clipping. На каждом шаге логируйте:
- норму градиента;
- максимальный логит attention \(\max_{i,j} q_i^\top k_j/\sqrt{d}\) по всем слоям;
- среднее отношение \(\|\Delta\theta\| / \|\theta\|\) за шаг (оценка через \(\eta\,\|\hat m/(\sqrt{\hat v}+\varepsilon)\|\));
- долю координат с очень малым \(\hat v\) (\(\sqrt{\hat v} < 10^{-8}\)).

Найдите шаги со всплесками loss и посмотрите, какие показатели меняются перед ними.

In [ ]:
captured = {}
def _hook(name):
    def f(mod, inp, out): captured[name] = inp[0].detach()
    return f
def attach_hooks(model):
    for i, b in enumerate(model.blocks): b.attn.register_forward_hook(_hook(i))

@torch.no_grad()
def max_attn_logit(model):
    # TODO: ваш код (из captured[i] (вход attention) пересчитайте q, k с RoPE и найдите максимум каузальных логитов)
    raise NotImplementedError

@torch.no_grad()
def diagnostics(model, opt, x, grad_norm, loss):
    rel_upd, tiny_v, n = 0.0, 0, 0
    for g in opt.param_groups:
        for p in g["params"]:
            st = opt.state.get(p, {})
            if "exp_avg" in st and st.get("step", 0):
                t = float(st["step"]); b1, b2 = g["betas"]
                m_hat = st["exp_avg"] / (1 - b1 ** t); v_hat = st["exp_avg_sq"] / (1 - b2 ** t)
                upd = g["lr"] * m_hat / (v_hat.sqrt() + g["eps"])
                rel_upd += (upd.norm() / (p.norm() + 1e-12)).item(); n += 1
                tiny_v += (v_hat.sqrt() < 1e-8).float().mean().item()
    return {"loss": loss, "grad_norm": grad_norm, "max_logit": max_attn_logit(model),
            "rel_update": rel_upd / max(n, 1), "tiny_v": tiny_v / max(n, 1)}

set_seed(); spike_model = GPT(SMALL).to(DEVICE); attach_hooks(spike_model)
_, h, val, diag = train_run(SMALL, "adamw", 10 * best_lr, 2 * SWEEP_STEPS, schedule="constant", warmup_frac=0.0,
                            log_diag=True, model=spike_model)
keys = ["loss", "grad_norm", "max_logit", "rel_update", "tiny_v"]
fig, axes = plt.subplots(len(keys), 1, figsize=(9, 10), sharex=True)
for ax, k in zip(axes, keys):
    ax.plot([d[k] for d in diag]); ax.set_ylabel(k)
axes[0].set_title(f"AdamW, LR = {10 * best_lr:g}, без warmup и clipping"); axes[-1].set_xlabel("шаг"); plt.tight_layout(); plt.show()

In [ ]:
# TODO: ваш код (найдите шаги, где loss вырос более чем на 20% относительно скользящего минимума, и выведите показатели за 5 шагов до всплеска)

## 5. Упрощённый μP по ширине

Сравните стандартную параметризацию (SP) и μP при ширинах \(d \in \{64, 128, 256\}\) (в полном режиме до 512) с фиксированной глубиной. Правила μP для Adam (Yang et al., 2022, упрощённо):
- скрытые матрицы (attention, FFN): инициализация \(\mathcal{N}(0, 1/d)\), learning rate \(\eta \cdot d_0/d\);
- эмбеддинги: обычный learning rate \(\eta\);
- выходной слой: не связан с эмбеддингами, инициализирован нулём, логиты умножаются на \(d_0/d\);
- attention: масштаб \(1/d_{head}\) вместо \(1/\sqrt{d_{head}}\) (в упрощённой версии можно оставить \(1/\sqrt{d_{head}}\) при фиксированном \(d_{head}\)).

Здесь \(d_0 = 64\) — базовая ширина. Для каждой параметризации и ширины найдите оптимальный LR и проверьте, сохраняется ли он при росте ширины.

In [ ]:
def make_mup(cfg, base_d=64):
    model = GPT(cfg).to(DEVICE); d = cfg.n_embd; mult = base_d / d
    # TODO: ваш код (отвяжите head от wte, обнулите его, добавьте forward hook, умножающий логиты на mult; инициализация скрытых матриц N(0, 1/d))
    raise NotImplementedError
    return model

def mup_groups(base_d=64):
    def f(model, lr):
        d = model.cfg.n_embd
        hidden = [p for n, p in model.named_parameters() if p.dim() == 2 and not n.startswith(("wte", "head"))]
        other = [p for n, p in model.named_parameters() if not (p.dim() == 2 and not n.startswith(("wte", "head")))]
        return [{"params": hidden, "lr": lr * base_d / d}, {"params": other, "lr": lr}]
    return f

WIDTHS = [64, 128, 256] if FAST_DEV_RUN else [64, 128, 256, 512]
MUP_LRS = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
# TODO: ваш код (sweep SP и μP по ширинам и LR, два графика val loss(LR) с кривой на каждую ширину)

## Вопросы

1. Вычислите величину первого шага Adam без bias correction при \(\beta_1 = 0.9\), \(\beta_2 = 0.999\) и сравните с шагом при bias correction. Объясните, почему warmup частично решает ту же проблему.
2. Почему L2-регуляризация в Adam не эквивалентна weight decay, а в SGD эквивалентна?
3. Какие показатели в п. 4 предшествуют всплескам? Предложите меры: какие из них устраняют причину, а какие — симптомы (QK-norm, z-loss, clipping, уменьшение \(\beta_2\), \(\varepsilon\))?
4. Почему WSD удобнее cosine для экспериментов со scaling laws и для дообучения?
5. Какое свойство μP позволяет переносить гиперпараметры с маленькой модели на большую? Что произошло с оптимальным LR в SP при росте ширины в вашем эксперименте?